# lluc — LoRA Rank vs Morphological Complexity

Fine-tune **mT5-small + LoRA** on verb inflection across three languages with different morphological richness, then sweep LoRA rank to find the **minimum rank required** for competent inflection in each language.

**Research question:** Does the required LoRA rank scale with paradigm complexity?

| Language | Paradigm size | Expected needed rank |
|----------|:-------------:|:--------------------:|
| English  | ~5 forms/verb | low |
| Spanish  | ~50 forms/verb | medium |
| Finnish  | ~150 forms/verb | high |

**Key design choices:**
- Sampling by **lemma** (not row) → fair cross-language comparison
- `lora_alpha = 2 × rank` → effective scale constant across rank sweep
- Fixed eval set (separate lemmas from train) → no data leakage


## 0. Install & GPU check

In [15]:
!pip install peft -q
!pip uninstall torchao -y -q  # Colab ships 0.10.0; peft needs 0.16.0+

import torch
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️  No GPU — training will be very slow")


GPU : Tesla T4
VRAM: 15.6 GB


## 1. Download UniMorph data

Run once per session. Downloads verb paradigm tables for English, Spanish, Finnish.


In [16]:
import requests
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# English and Spanish: single file each
SINGLE_URLS = {
    "eng": "https://raw.githubusercontent.com/unimorph/eng/master/eng",
    "spa": "https://raw.githubusercontent.com/unimorph/spa/master/spa",
}

# Finnish is published as two parts; we concatenate them into fin.tsv
FIN_PART_URLS = [
    "https://raw.githubusercontent.com/unimorph/fin/master/fin.1",
    "https://raw.githubusercontent.com/unimorph/fin/master/fin.2",
]

def _download(url, dest, label):
    print(f"  {label}: downloading...", end="", flush=True)
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f" done ({dest.stat().st_size//1024} KB)")

# Download single-file languages
for lang, url in SINGLE_URLS.items():
    dest = DATA_DIR / f"{lang}.tsv"
    if dest.exists():
        print(f"  {lang}: already present ({dest.stat().st_size//1024} KB)")
    else:
        _download(url, dest, lang)

# Download Finnish (two parts → concatenate)
fin_dest = DATA_DIR / "fin.tsv"
if fin_dest.exists():
    print(f"  fin: already present ({fin_dest.stat().st_size//1024} KB)")
else:
    parts = []
    for i, url in enumerate(FIN_PART_URLS, 1):
        part = DATA_DIR / f"fin.part{i}"
        _download(url, part, f"fin.{i}")
        parts.append(part)
    print("  fin: concatenating fin.1 + fin.2 ...", end="", flush=True)
    with open(fin_dest, "wb") as out:
        for p in parts:
            out.write(p.read_bytes())
            p.unlink()
    print(f" done ({fin_dest.stat().st_size//1024} KB)")

print("\nAll data ready:")
for f in sorted(DATA_DIR.glob("*.tsv")):
    print(f"  {f.name}  {f.stat().st_size/1e6:.1f} MB")


  eng: already present (17600 KB)
  spa: already present (49156 KB)
  fin: already present (112887 KB)

All data ready:
  eng.tsv  18.0 MB
  fin.tsv  115.6 MB
  spa.tsv  50.3 MB


## 2. Helpers

All functions — run once per session.


In [17]:
import os, time, json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"]         = "true"

DATA_DIR    = Path("/content/data")
RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(exist_ok=True)

# ── Data loading ───────────────────────────────────────────────────────────

def load_unimorph(tsv_path, pos="V"):
    """Load UniMorph TSV and filter to the requested POS tag."""
    df = pd.read_csv(tsv_path, sep="\t", header=None,
                     names=["lemma","form","features"], dtype=str, na_filter=False)
    if pos:
        df = df[df["features"].str.startswith(pos+";")].reset_index(drop=True)
    n_lemmas = df["lemma"].nunique()
    avg_forms = len(df) / n_lemmas if n_lemmas else 0
    print(f"    {len(df):>9,} rows | {n_lemmas:>6,} lemmas | {avg_forms:.0f} avg forms/lemma")
    return df

def split_by_lemma(df, n_train_lemmas, n_eval_lemmas, max_train_rows=2000, seed=42):
    """
    Split dataset by LEMMA — train and eval never share a paradigm.
    This ensures a fair cross-language comparison: each language gets the
    same number of paradigms, regardless of how many surface forms each has.

    Args:
        n_train_lemmas : lemmas reserved for training
        n_eval_lemmas  : lemmas reserved for evaluation
        max_train_rows : cap training rows (Finnish has 150+ forms/lemma; without
                         this cap, Finnish would take 30x longer per epoch)
    """
    rng = np.random.default_rng(seed)
    all_lemmas = df["lemma"].unique().copy()
    rng.shuffle(all_lemmas)
    eval_l  = set(all_lemmas[:n_eval_lemmas])
    train_l = set(all_lemmas[n_eval_lemmas : n_eval_lemmas + n_train_lemmas])
    df_eval  = df[df["lemma"].isin(eval_l)].reset_index(drop=True)
    df_train = df[df["lemma"].isin(train_l)].reset_index(drop=True)
    if len(df_train) > max_train_rows:
        df_train = df_train.sample(max_train_rows, random_state=seed).reset_index(drop=True)
    return df_train, df_eval

def build_valid_forms(df):
    """Build {(lemma, features): set(valid surface forms)} from the full dataset.
    Used by multi-valid exact-match evaluation."""
    groups = df.groupby(["lemma","features"])["form"].apply(set)
    return {k: v for k, v in groups.items()}

# ── Pair formatting ─────────────────────────────────────────────────────────

def make_pairs(df):
    """
    Sentinel format: <extra_id_0> in input and target prefix.
    Matches mT5 span-corruption pre-training so the untrained model uses
    morphological knowledge rather than defaulting to '<extra_id_0>'.
    Used for C2 (pretrained baseline) and C3 (LoRA fine-tuned).
    """
    return [{"input":        f"inflect: {r['lemma']} | {r['features']} <extra_id_0>",
             "target":       f"<extra_id_0> {r['form']}",
             "clean_target": r["form"],
             "lemma":        r["lemma"],
             "features":     r["features"]}
            for _, r in df.iterrows()]

def make_pairs_plain(df):
    """Plain format — no sentinel. Used for C1 (illustrates format confusion)."""
    return [{"input":        f"inflect: {r['lemma']} | {r['features']}",
             "target":       r["form"],
             "clean_target": r["form"],
             "lemma":        r["lemma"],
             "features":     r["features"]}
            for _, r in df.iterrows()]

# ── Model ───────────────────────────────────────────────────────────────────

def build_lora_model(model_name="google/mt5-small", lora_r=4):
    """
    Build mT5-small + LoRA adapter.

    lora_alpha = 2 * lora_r  (standard convention)
    → effective scaling factor (alpha/r) = 2, constant across rank sweep.
    This ensures rank comparisons are not confounded by different LR scales.
    """
    tokenizer  = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    cfg = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=lora_r,
        lora_alpha=lora_r * 2,    # keep scale constant: alpha/r = 2
        target_modules=["q", "v"],
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(base_model, cfg)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return model, tokenizer, n_params

# ── Dataset + training ──────────────────────────────────────────────────────

class MorphDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len=48):
        self.pairs   = pairs
        self.tok     = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.pairs)

    def __getitem__(self, i):
        p   = self.pairs[i]
        enc = self.tok(p["input"],  truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        dec = self.tok(p["target"], truncation=True, max_length=16,
                       padding="max_length", return_tensors="pt")
        labels = dec["input_ids"].squeeze()
        labels[labels == self.tok.pad_token_id] = -100
        return {"input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "labels":         labels}

def train_epoch(model, loader, optimizer, device, scheduler=None):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        optimizer.zero_grad()
        loss = model(input_ids=batch["input_ids"].to(device),
                     attention_mask=batch["attention_mask"].to(device),
                     labels=batch["labels"].to(device)).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total += loss.item(); n += 1
    return total / max(n, 1)

# ── Evaluation ──────────────────────────────────────────────────────────────

def _edit_distance(s, t):
    """Levenshtein distance (no external dependency)."""
    m, n = len(s), len(t)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            prev, dp[j] = dp[j], (prev if s[i-1]==t[j-1]
                                   else 1 + min(prev, dp[j], dp[j-1]))
    return dp[n]

def char_acc(pred, gold):
    """Character-level accuracy: 1 - edit_distance/len(gold). Secondary metric."""
    if not gold: return 1.0 if not pred else 0.0
    return max(0.0, 1.0 - _edit_distance(pred, gold) / len(gold))

@torch.no_grad()
def evaluate(model, tokenizer, eval_pairs, valid_forms, device,
             max_len=48, gen_max=24, n_examples=3):
    """
    Evaluate morphological inflection.

    PRIMARY metric   → multi_em : prediction ∈ all valid forms for (lemma, features)
    SECONDARY metric → strict_em: prediction == gold exactly
    DIAGNOSTIC       → char_acc : partial character credit
    """
    model.eval()
    strict_ok = multi_ok = char_sum = total = 0
    examples  = []
    for p in eval_pairs:
        enc = tokenizer(p["input"], return_tensors="pt",
                        truncation=True, max_length=max_len)
        try:
            gen  = model.generate(input_ids=enc["input_ids"].to(device),
                                   attention_mask=enc["attention_mask"].to(device),
                                   max_new_tokens=gen_max)
            pred = tokenizer.decode(gen[0], skip_special_tokens=True).strip()
            if pred.startswith("<extra_id_0>"):
                pred = pred[len("<extra_id_0>"):].strip()
        except Exception:
            pred = ""
        gold      = p.get("clean_target", p["target"]).strip()
        all_valid = valid_forms.get((p["lemma"], p["features"]), {gold})
        ca        = char_acc(pred, gold)
        strict_ok += int(pred == gold)
        multi_ok  += int(pred in all_valid)
        char_sum  += ca; total += 1
        if len(examples) < n_examples:
            examples.append({"input": p["input"], "pred": pred, "gold": gold,
                              "strict": pred==gold, "multi": pred in all_valid,
                              "char_acc": round(ca,3), "n_valid": len(all_valid)})
    n = max(total, 1)
    return 100*strict_ok/n, 100*multi_ok/n, 100*char_sum/n, examples

def copy_lemma_eval(eval_pairs, valid_forms):
    """Condition 4: always predict the lemma — trivial morphological baseline."""
    strict = multi = cacc_sum = total = 0
    for p in eval_pairs:
        pred      = p["lemma"]
        gold      = p.get("clean_target", p["target"]).strip()
        all_valid = valid_forms.get((p["lemma"], p["features"]), {gold})
        strict   += int(pred == gold)
        multi    += int(pred in all_valid)
        cacc_sum += char_acc(pred, gold); total += 1
    n = max(total, 1)
    return 100*strict/n, 100*multi/n, 100*cacc_sum/n

print("Helpers loaded ✓")


Helpers loaded ✓


## 3. Config

**Edit only this cell**, then Runtime → Run all.


In [18]:
# ════════════════════════════════════════════════════════════════════════════
#  CONFIG  — all experiment parameters in one place
# ════════════════════════════════════════════════════════════════════════════

LANGS           = ["eng", "spa", "fin"]   # languages to compare

# ── Sampling (lemma-based for fair cross-language comparison) ─────────────
N_TRAIN_LEMMAS  = 500    # training paradigms per language
N_EVAL_LEMMAS   = 50     # evaluation paradigms per language
MAX_TRAIN_ROWS  = 4000   # cap training rows (Finnish has ~150 forms/lemma)
EVAL_N          = 200    # cap evaluation examples

# ── Rank sweep (main experiment) ─────────────────────────────────────────
LORA_RANKS      = [1, 2, 4, 8, 16, 32]   # LoRA ranks to test

# ── Training ─────────────────────────────────────────────────────────────
EPOCHS          = 6      # epochs per (language, rank) run
BATCH           = 8
LR              = 2e-4

# ── Minimum-rank threshold ────────────────────────────────────────────────
TARGET_ACC      = 50.0   # % — "minimum rank to reach TARGET_ACC multi_em"

# ── Model ─────────────────────────────────────────────────────────────────
MODEL_NAME      = "google/mt5-small"
POS             = "V"    # verb entries only
SEED            = 42
MAX_LEN         = 48
GEN_MAX         = 24

# ─────────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

n_runs = len(LANGS) * len(LORA_RANKS)
steps_per_epoch = MAX_TRAIN_ROWS // BATCH
est_min = n_runs * EPOCHS * steps_per_epoch * 3 // 1000   # rough T4 estimate
print(f"device={DEVICE}   langs={LANGS}")
print(f"train_lemmas={N_TRAIN_LEMMAS}  eval_lemmas={N_EVAL_LEMMAS}  max_train_rows={MAX_TRAIN_ROWS}")
print(f"ranks={LORA_RANKS}  epochs={EPOCHS}  lr={LR}")
print(f"Total runs: {n_runs}  |  Est. T4 time: ~{est_min}–{est_min+20} min")


device=cuda   langs=['eng', 'spa', 'fin']
train_lemmas=500  eval_lemmas=50  max_train_rows=4000
ranks=[1, 2, 4, 8, 16, 32]  epochs=6  lr=0.0002
Total runs: 18  |  Est. T4 time: ~162–182 min


## 4. Run experiment

For each language:
1. **C1** — pretrained mT5, plain format (format-confusion floor)
2. **C2** — pretrained mT5, sentinel format (actual zero-shot morphological knowledge)
3. **C4** — copy-lemma oracle (trivial baseline)
4. **C3** — LoRA fine-tuned at each rank in `LORA_RANKS`

Results are saved to `/content/results/` after each language.


In [ ]:
results_cond = []   # 4-condition table
results_rank = []   # rank sweep: (lang, rank, multi_em, strict_em, char_acc, loss)
loss_curves  = []   # (lang, rank, epoch, train_loss)

for lang in LANGS:
    tsv = DATA_DIR / f"{lang}.tsv"
    print(f"\n{'='*60}")
    print(f"  Language: {lang.upper()}")
    print(f"{'='*60}")

    # ── Load & split ──────────────────────────────────────────────────
    print("  Loading data...")
    df_full = load_unimorph(tsv, pos=POS)
    valid_forms = build_valid_forms(df_full)

    df_train, df_eval = split_by_lemma(
        df_full, N_TRAIN_LEMMAS, N_EVAL_LEMMAS,
        max_train_rows=MAX_TRAIN_ROWS, seed=SEED)

    # Cap eval examples
    if len(df_eval) > EVAL_N:
        df_eval = df_eval.sample(EVAL_N, random_state=SEED).reset_index(drop=True)

    eval_sentinel = make_pairs(df_eval)
    eval_plain    = make_pairs_plain(df_eval)

    print(f"  Train: {len(df_train)} rows ({df_train['lemma'].nunique()} lemmas)  "
          f"Eval: {len(df_eval)} rows ({df_eval['lemma'].nunique()} lemmas)")

    # ── Baselines (no training needed) ───────────────────────────────
    print("\n  [C1  plain-format pretrained]")
    tok_base  = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    model_base = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    model_base.eval()

    c1_s, c1_m, c1_c, _ = evaluate(model_base, tok_base, eval_plain,
                                     valid_forms, DEVICE, MAX_LEN, GEN_MAX)
    print(f"    multi={c1_m:.1f}%  strict={c1_s:.1f}%  char={c1_c:.1f}%  (format confusion floor)")

    c2_s, c2_m, c2_c, _ = evaluate(model_base, tok_base, eval_sentinel,
                                     valid_forms, DEVICE, MAX_LEN, GEN_MAX)
    print(f"  [C2  sentinel pretrained]")
    print(f"    multi={c2_m:.1f}%  strict={c2_s:.1f}%  char={c2_c:.1f}%  (real mT5 knowledge)")

    c4_s, c4_m, c4_c = copy_lemma_eval(eval_sentinel, valid_forms)
    print(f"  [C4  copy-lemma oracle]")
    print(f"    multi={c4_m:.1f}%  strict={c4_s:.1f}%  char={c4_c:.1f}%")

    del model_base; torch.cuda.empty_cache()

    results_cond.extend([
        {"lang": lang, "condition": "C1 — Pretrained (plain)",
         "multi_em": round(c1_m,2), "strict_em": round(c1_s,2), "char_acc": round(c1_c,2),
         "notes": "format confusion floor"},
        {"lang": lang, "condition": "C2 — Pretrained (sentinel)",
         "multi_em": round(c2_m,2), "strict_em": round(c2_s,2), "char_acc": round(c2_c,2),
         "notes": "actual mT5 knowledge"},
        {"lang": lang, "condition": "C4 — Copy-lemma oracle",
         "multi_em": round(c4_m,2), "strict_em": round(c4_s,2), "char_acc": round(c4_c,2),
         "notes": "trivial baseline"},
    ])

    # ── Rank sweep (C3) ───────────────────────────────────────────────
    train_pairs = make_pairs(df_train)

    best_rank_result = None
    for lora_r in LORA_RANKS:
        print(f"\n  [C3  LoRA r={lora_r}]")
        model3, tok3, n_params = build_lora_model(MODEL_NAME, lora_r)
        model3 = model3.to(DEVICE)
        print(f"    trainable params: {n_params:,}  |  alpha={lora_r*2}")

        loader = DataLoader(MorphDataset(train_pairs, tok3, MAX_LEN),
                            batch_size=BATCH, shuffle=True, num_workers=0)

        optimizer = AdamW(filter(lambda p: p.requires_grad, model3.parameters()), lr=LR)
        total_steps  = EPOCHS * len(loader)
        warmup_steps = max(1, total_steps // 10)
        warmup_sched = LinearLR(optimizer, start_factor=0.1, end_factor=1.0,
                                 total_iters=warmup_steps)
        decay_sched  = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps,
                                          eta_min=LR * 0.1)
        scheduler    = SequentialLR(optimizer, [warmup_sched, decay_sched],
                                     milestones=[warmup_steps])

        t0 = time.time(); last_loss = float("nan")
        for epoch in range(1, EPOCHS + 1):
            last_loss = train_epoch(model3, loader, optimizer, DEVICE, scheduler)
            loss_curves.append({"lang": lang, "rank": lora_r,
                                 "epoch": epoch, "train_loss": round(last_loss, 6)})

        c3_s, c3_m, c3_c, examples = evaluate(
            model3, tok3, eval_sentinel, valid_forms, DEVICE, MAX_LEN, GEN_MAX)
        elapsed = round(time.time() - t0, 1)

        print(f"    multi={c3_m:.1f}%  strict={c3_s:.1f}%  char={c3_c:.1f}%  "
              f"loss={last_loss:.4f}  ({elapsed:.0f}s)")
        print(f"    Δ over C2: multi+{c3_m-c2_m:.1f}pp  strict+{c3_s-c2_s:.1f}pp")

        # Print examples only for the largest rank
        if lora_r == LORA_RANKS[-1]:
            print("    examples (largest rank):")
            for ex in examples:
                mark = "✓" if ex["strict"] else ("~" if ex["multi"] else "✗")
                print(f"      {mark} pred={ex['pred']!r:20s}  gold={ex['gold']!r}"
                      f"  char={ex['char_acc']:.2f}  ({ex['n_valid']} valid forms)")
            best_rank_result = {"C3_multi": round(c3_m,2), "C3_strict": round(c3_s,2),
                                 "C3_char": round(c3_c,2), "C3_rank": lora_r}

        results_rank.append({"lang": lang, "rank": lora_r,
                              "multi_em": round(c3_m,2), "strict_em": round(c3_s,2),
                              "char_acc": round(c3_c,2), "final_loss": round(last_loss,4),
                              "seconds": elapsed, "n_params": n_params})

        del model3; torch.cuda.empty_cache()

    # Add best C3 to conditions table
    if best_rank_result:
        results_cond.append({
            "lang": lang,
            "condition": f"C3 — LoRA r={best_rank_result['C3_rank']} (best)",
            "multi_em": best_rank_result["C3_multi"],
            "strict_em": best_rank_result["C3_strict"],
            "char_acc": best_rank_result["C3_char"],
            "notes": "LoRA fine-tuned",
        })

    # Save after each language
    pd.DataFrame(results_cond).to_csv(RESULTS_DIR/"conditions.csv", index=False)
    pd.DataFrame(results_rank).to_csv(RESULTS_DIR/"rank_sweep.csv", index=False)
    pd.DataFrame(loss_curves).to_csv(RESULTS_DIR/"loss_curves.csv", index=False)
    print(f"\n  Saved results for {lang.upper()} ✓")

print("\n" + "="*60)
print("  ALL DONE")
print("="*60)



  Language: ENG
  Loading data...
      127,514 rows | 31,850 lemmas | 4 avg forms/lemma
  Train: 1996 rows (500 lemmas)  Eval: 200 rows (50 lemmas)

  [C1  plain-format pretrained]


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


    multi=0.0%  strict=0.0%  char=0.0%  (format confusion floor)
  [C2  sentinel pretrained]
    multi=0.0%  strict=0.0%  char=0.0%  (real mT5 knowledge)
  [C4  copy-lemma oracle]
    multi=24.5%  strict=24.5%  char=84.8%

  [C3  LoRA r=1]


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


    trainable params: 688,128  |  alpha=32
    multi=28.5%  strict=28.5%  char=85.3%  loss=1.5235  (179s)
    Δ over C2: multi+28.5pp  strict+28.5pp

  [C3  LoRA r=32]


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


    trainable params: 1,376,256  |  alpha=64


## 5b. Diagnostic: C5 — Natural Language Instructions (pretrained only)

**Research question:** Is the low baseline (C2 = 0%) due to format confusion (model can't parse `V;PST;3;SG`) or genuine lack of morphological knowledge?

**Method:** Run the *pretrained* mT5 with natural-language feature descriptions instead of UniMorph tags. No fine-tuning — identical to C2 except for prompt format.

| Result | Interpretation |
|--------|---------------|
| C5 >> C2 (0%) | Format IS the bottleneck — switch to NL prompts for full experiment |
| C5 ≈ C2 ≈ 0% | Model lacks the morphological knowledge regardless of format |


In [ ]:
# ── Feature tag → natural language mapping ────────────────────────────────
FEATURE_TO_NL = {
    # POS (skipped in output)
    "V": "",
    # Tense
    "PST": "past tense",        "PRS": "present tense",
    "FUT": "future tense",      "FUTR": "future tense",
    # Person
    "1": "first person",        "2": "second person",       "3": "third person",
    # Number
    "SG": "singular",           "PL": "plural",
    # Mood
    "IND": "indicative",        "SBJV": "subjunctive",      "IMP": "imperative",
    "COND": "conditional",      "OPT": "optative",
    # Aspect
    "IPFV": "imperfective",     "PFV": "perfective",
    "PROG": "progressive",      "PRF": "perfect",
    # Voice
    "ACT": "active",            "PASS": "passive",
    # Verbforms
    "INF": "infinitive",        "PTCP": "participle",
    "GER": "gerund",            "CONV": "converb",
}

def tag_to_nl(features_str):
    """Convert 'V;PST;3;SG' → 'past tense third person singular'."""
    parts = []
    for f in features_str.split(";"):
        nl = FEATURE_TO_NL.get(f)
        if nl is None:
            parts.append(f.lower())   # keep unknown tags as-is (lowercase)
        elif nl:                       # skip empty string (e.g. "V")
            parts.append(nl)
    return " ".join(parts)

def make_pairs_nl(df):
    """
    Natural-language format — no UniMorph tags in the prompt.
    Input:  'Conjugate \'walk\' in past tense third person singular: <extra_id_0>'
    Target: '<extra_id_0> walked'
    Used for C5 (pretrained NL) and optionally for C3-NL fine-tuning.
    """
    return [{"input":        f"Conjugate '{r['lemma']}' in {tag_to_nl(r['features'])}: <extra_id_0>",
             "target":       f"<extra_id_0> {r['form']}",
             "clean_target": r["form"],
             "lemma":        r["lemma"],
             "features":     r["features"]}
            for _, r in df.iterrows()]

# ── C5: pretrained mT5 with natural-language prompts ─────────────────────
print("C5 — Pretrained mT5 with natural-language feature descriptions")
print("(no fine-tuning — same model as C2, different prompt format)\n")

results_c5 = []

for lang in LANGS:
    tsv = DATA_DIR / f"{lang}.tsv"
    df_full = load_unimorph(tsv, pos=POS)
    valid_forms = build_valid_forms(df_full)

    # Use the same eval split as the main experiment
    _, df_eval = split_by_lemma(df_full, N_TRAIN_LEMMAS, N_EVAL_LEMMAS,
                                 max_train_rows=MAX_TRAIN_ROWS, seed=SEED)
    if len(df_eval) > EVAL_N:
        df_eval = df_eval.sample(EVAL_N, random_state=SEED).reset_index(drop=True)

    eval_nl = make_pairs_nl(df_eval)

    # Show a few prompts so you can verify the format makes sense
    print(f"  [{lang.upper()}] example prompts:")
    for p in eval_nl[:3]:
        print(f"    input : {p['input']}")
        print(f"    target: {p['clean_target']}\n")

    # Load pretrained model
    tok   = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

    c5_s, c5_m, c5_c, examples = evaluate(model, tok, eval_nl, valid_forms,
                                            DEVICE, MAX_LEN, GEN_MAX, n_examples=5)
    del model; torch.cuda.empty_cache()

    print(f"  [{lang.upper()}]  C5 NL-pretrained: multi={c5_m:.1f}%  strict={c5_s:.1f}%  char={c5_c:.1f}%")
    print(f"  [{lang.upper()}]  C2 tag-pretrained: multi=0.0%  (from main experiment)")
    verdict = "FORMAT IS bottleneck" if c5_m > 2.0 else "KNOWLEDGE IS bottleneck"
    print(f"  [{lang.upper()}]  → {verdict}\n")

    print(f"  [{lang.upper()}] examples:")
    for ex in examples:
        mark = "✓" if ex["multi"] else "✗"
        print(f"    {mark} pred={ex['pred']!r:25s}  gold={ex['gold']!r}  "
              f"(input was: {ex['input'][:60]}...)")

    results_c5.append({"lang": lang, "C5_multi": round(c5_m,2),
                        "C5_strict": round(c5_s,2), "C5_char": round(c5_c,2)})
    print()

# Summary table
import pandas as pd
df_c5 = pd.DataFrame(results_c5)
df_c5["C2_multi"] = 0.0
df_c5["gap"] = df_c5["C5_multi"] - df_c5["C2_multi"]
df_c5["interpretation"] = df_c5["gap"].apply(
    lambda g: "format bottleneck" if g > 2.0 else "knowledge bottleneck")
print("\n=== C5 vs C2 Summary ===")
display(df_c5[["lang","C2_multi","C5_multi","gap","interpretation"]].set_index("lang"))


## 5. 4-condition results table

C1 and C2 show what the pretrained mT5 already knows.  
C4 is the trivial copy-lemma baseline.  
**C3 at the highest rank is our main result. Δ = C3 − C2 = pure morphological adaptation.**


In [ ]:
from pathlib import Path
import pandas as pd

cond_path = RESULTS_DIR / "conditions.csv"
if not cond_path.exists():
    print("Not found — run the experiment cell first.")
else:
    df_c = pd.read_csv(cond_path)
    display(df_c.style
            .format({"multi_em":   "{:.1f}",
                     "strict_em":  "{:.1f}",
                     "char_acc":   "{:.1f}"})
            .highlight_max(subset=["multi_em","strict_em","char_acc"], axis=0,
                           color="lightgreen")
            .set_caption("4-Condition results — primary metric: Multi-Valid EM%"))

    # Print clean delta summary
    print("\n--- Δ (C3 − C2): pure morphological adaptation ---\n")
    for lang in df_c["lang"].unique():
        sub  = df_c[df_c["lang"]==lang].set_index("condition")
        c2   = sub.get("C2 — Pretrained (sentinel)", sub.iloc[1:2])
        c3   = sub[sub.index.str.startswith("C3")]
        if c2.empty or c3.empty: continue
        c2r  = c2.iloc[0]
        c3r  = c3.iloc[-1]
        print(f"  {lang.upper():<5}  "
              f"multi: {c2r['multi_em']:.1f}% → {c3r['multi_em']:.1f}%  "
              f"(+{c3r['multi_em']-c2r['multi_em']:.1f}pp)  |  "
              f"strict: {c2r['strict_em']:.1f}% → {c3r['strict_em']:.1f}%  "
              f"(+{c3r['strict_em']-c2r['strict_em']:.1f}pp)")


## 6. Rank sweep: accuracy vs LoRA rank

**Main result plot.** Each line is a language; x-axis is LoRA rank (log scale).  
- A language that saturates early needs low rank (simple morphology)
- A language still rising at r=32 needs higher rank (rich morphology)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

rank_path = RESULTS_DIR / "rank_sweep.csv"
if not rank_path.exists():
    print("Not found — run the experiment cell first.")
else:
    df_r = pd.read_csv(rank_path)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    colors = {"eng": "#2196F3", "spa": "#F44336", "fin": "#4CAF50"}
    markers = {"eng": "o", "spa": "s", "fin": "^"}

    for metric, ax, title in [
        ("multi_em",  axes[0], "Multi-Valid Exact Match %  (PRIMARY)"),
        ("char_acc",  axes[1], "Character Accuracy %  (diagnostic)"),
    ]:
        for lang in df_r["lang"].unique():
            sub = df_r[df_r["lang"]==lang].sort_values("rank")
            ax.plot(sub["rank"], sub[metric],
                    label=lang.upper(), color=colors.get(lang,"grey"),
                    marker=markers.get(lang,"o"), linewidth=2, markersize=7)

        ax.set_xscale("log", base=2)
        ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
        ax.set_xticks(df_r["rank"].unique())
        ax.set_xlabel("LoRA rank  (log₂ scale)", fontsize=12)
        ax.set_ylabel(title, fontsize=12)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(bottom=0)

    plt.suptitle("LoRA Rank vs Morphological Inflection Accuracy", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "rank_sweep.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → rank_sweep.png")


## 7. Minimum rank analysis

For each language: what is the **lowest rank that reaches TARGET_ACC%** multi-valid EM?  
This gives a single number per language that can be correlated with morphological complexity.


In [ ]:
rank_path = RESULTS_DIR / "rank_sweep.csv"
if not rank_path.exists():
    print("Not found — run the experiment cell first.")
else:
    df_r = pd.read_csv(rank_path)
    print(f"Minimum LoRA rank to reach {TARGET_ACC:.0f}% multi-valid EM:\n")
    summary = []
    for lang in LANGS:
        sub = df_r[df_r["lang"]==lang].sort_values("rank")
        above = sub[sub["multi_em"] >= TARGET_ACC]
        if above.empty:
            min_rank = ">32"
            best_acc = sub["multi_em"].max()
            print(f"  {lang.upper():<5}  never reached {TARGET_ACC:.0f}%  "
                  f"(best: {best_acc:.1f}%  at r={sub.loc[sub['multi_em'].idxmax(),'rank']})")
        else:
            min_rank = int(above.iloc[0]["rank"])
            acc_at   = above.iloc[0]["multi_em"]
            print(f"  {lang.upper():<5}  min rank = {min_rank:>2}  "
                  f"(multi_em = {acc_at:.1f}% at r={min_rank})")
        summary.append({"lang": lang.upper(), "min_rank_for_target": min_rank,
                         "target_acc": TARGET_ACC,
                         "best_multi_em": round(sub['multi_em'].max(),2)})

    print("\n")
    display(pd.DataFrame(summary).set_index("lang"))

    # ── Paradigm size vs min rank plot ────────────────────────────────
    # Approximate paradigm sizes from UniMorph
    paradigm_sizes = {"eng": 5, "spa": 50, "fin": 150}
    fig, ax = plt.subplots(figsize=(7, 4))
    for row in summary:
        lang = row["lang"].lower()
        mr   = row["min_rank_for_target"]
        if isinstance(mr, int):
            ps = paradigm_sizes.get(lang, 1)
            ax.scatter(ps, mr, s=150, color=colors.get(lang,"grey"),
                       label=lang.upper(), zorder=5)
            ax.annotate(f"  {lang.upper()}  r={mr}", (ps, mr), fontsize=11)

    ax.set_xscale("log")
    ax.set_xlabel("Paradigm size (forms/lemma, log scale)", fontsize=12)
    ax.set_ylabel(f"Min LoRA rank for {TARGET_ACC:.0f}% accuracy", fontsize=12)
    ax.set_title("Morphological complexity vs required LoRA rank", fontsize=13, fontweight="bold")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "min_rank_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → min_rank_analysis.png")


## 8. Training loss curves

Loss per epoch for each (language, rank) combination.  
All ranks should converge; higher rank may converge to a lower loss.


In [ ]:
loss_path = RESULTS_DIR / "loss_curves.csv"
if not loss_path.exists():
    print("Not found — run the experiment cell first.")
else:
    df_l = pd.read_csv(loss_path)
    langs = df_l["lang"].unique()
    fig, axes = plt.subplots(1, len(langs), figsize=(5*len(langs), 4), sharey=False)
    if len(langs)==1: axes = [axes]

    rank_colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(LORA_RANKS)))

    for ax, lang in zip(axes, langs):
        for i, r in enumerate(sorted(df_l["rank"].unique())):
            sub = df_l[(df_l["lang"]==lang) & (df_l["rank"]==r)].sort_values("epoch")
            ax.plot(sub["epoch"], sub["train_loss"],
                    label=f"r={r}", color=rank_colors[i], linewidth=1.8)
        ax.set_title(lang.upper(), fontsize=13, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Train loss")
        ax.legend(fontsize=9, title="LoRA rank"); ax.grid(True, alpha=0.3)

    plt.suptitle("Training loss by language and LoRA rank", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → loss_curves.png")


## Notes

### Experimental design

| Condition | Input format | Purpose |
|-----------|-------------|---------|
| C1 — plain pretrained | `inflect: {lemma} \| {features}` | Format-confusion floor (model outputs `<extra_id_0>`) |
| C2 — sentinel pretrained | `inflect: {lemma} \| {features} <extra_id_0>` | Actual mT5 zero-shot morphological knowledge |
| C3 — LoRA fine-tuned | sentinel format | Main result; swept over ranks `[1,2,4,8,16,32]` |
| C4 — copy-lemma | — | Trivial baseline (predict lemma unchanged) |

**Δ (C3 − C2) = purely morphological learning by LoRA** (format confound removed by sentinel).

### Key methodological choices

- **Lemma-based split**: train and eval sets share no lemmas — tests generalisation, not memorisation
- **`lora_alpha = 2 × r`**: effective LoRA scale stays constant as rank varies, so rank sweeps are not confounded by implicit LR differences
- **`max_train_rows` cap**: Finnish has ~150 forms/lemma; without a cap it would take 30× longer per epoch than English — capping rows keeps training time comparable while preserving the lemma-count fairness
- **Multi-valid exact match (primary)**: some UniMorph slots have multiple dialect variants; a prediction is correct if it matches **any** listed valid form for the (lemma, features) slot

### Interpreting the rank sweep

```
Multi-Valid EM
│
│ ●──●──●──●──●──● ← ENG: saturates early (low rank sufficient)
│         ●──●──●──●──● ← SPA: needs medium rank
│                  ●──●──●──● ← FIN: still rising at high rank
└───────────────────────── LoRA rank
  1   2   4   8   16  32
```

The minimum rank to reach target accuracy is a **single number per language**
that directly quantifies how morphological complexity translates into model capacity requirements.
